# Arquitectura de una Red Neuronal

Hasta ahora estudiamos una sola neurona:

\[
a=f(\mathbf{w}\cdot\mathbf{x}+b).
\]

El verdadero poder de las redes neuronales aparece cuando conectamos **muchas neuronas en capas**.

## Objetivos

Al finalizar podrás:

- identificar input, hidden y output layers;
- entender qué significa el tamaño de una capa;
- seguir las dimensiones de los datos a través de una red;
- calcular cuántos parámetros tiene una capa;
- construir una red pequeña con NumPy;
- interpretar una arquitectura a partir de un diagrama.


In [ ]:
import numpy as np


## 1. De una neurona a una capa

Una neurona produce una salida.

Una **capa** contiene varias neuronas.

```text
x₁ ─────┐        ○
x₂ ─────┼──────► ○
x₃ ─────┤        ○
x₄ ─────┘        ○
              Hidden Layer
```

Si tenemos 4 features y 5 neuronas:

```text
Input size  = 4
Output size = 5
```

Cada neurona recibe las 4 features.


## 2. Input Layer

El **input layer** representa las features.

Por ejemplo, en Iris:

```text
x₁ = sepal length
x₂ = sepal width
x₃ = petal length
x₄ = petal width
```

Tenemos:

\[
n_{\mathrm{features}}=4.
\]

Por lo tanto, la red debe aceptar cuatro valores por sample.


## 3. Hidden Layer

Supongamos que queremos una hidden layer con 6 neuronas.

Cada neurona necesita:

- 4 weights;
- 1 bias.

Entonces:

```text
4 inputs
   ↓
6 hidden neurons
```

La matriz de pesos tendrá forma:

\[
W\in\mathbb{R}^{6\times 4}.
\]

Y el vector de biases:

\[
b\in\mathbb{R}^{6}.
\]


In [ ]:
n_inputs = 4
n_neurons = 6

W = np.random.randn(n_neurons, n_inputs)
b = np.random.randn(n_neurons)

print("W.shape =", W.shape)
print("b.shape =", b.shape)


### Pregunta

¿Cuántos weights hay en esta capa?

<details>
<summary><strong>Pista</strong></summary>

Multiplica:

\[
6\times4
\]

</details>

<details>
<summary><strong>Mostrar solución</strong></summary>

Hay:

\[
6\times4=24
\]

weights.

Además hay 6 biases.

Total:

\[
24+6=30
\]

parámetros entrenables.

</details>


## 4. Forward pass de una capa

Para un sample:

\[
\mathbf{x}\in\mathbb{R}^{4}
\]

podemos calcular:

\[
\mathbf{z}=W\mathbf{x}+b.
\]

La salida tendrá 6 valores porque la capa tiene 6 neuronas.


In [ ]:
x = np.array([5.1, 3.5, 1.4, 0.2])

z = W @ x + b

print("input shape :", x.shape)
print("output shape:", z.shape)
print("z =", z)


Aplicamos una activación elemento por elemento:


In [ ]:
def relu(z):
    return np.maximum(0, z)

a = relu(z)

print("activations =", a)


## 5. Añadiendo una output layer

Iris tiene tres especies:

```text
Setosa
Versicolor
Virginica
```

Podemos crear una red:

```text
4 inputs
   ↓
6 hidden neurons
   ↓
3 outputs
```

La segunda capa necesita una matriz:

\[
W_2\in\mathbb{R}^{3\times6}.
\]


In [ ]:
W2 = np.random.randn(3, 6)
b2 = np.random.randn(3)

output = W2 @ a + b2

print("hidden activation shape:", a.shape)
print("output shape:", output.shape)
print("output:", output)


Los tres valores finales pueden llamarse **logits**.

Más adelante aprenderemos cómo convertirlos en probabilidades usando **Softmax**.


## 6. Diagrama completo

Nuestra arquitectura es:

```text
INPUT LAYER       HIDDEN LAYER       OUTPUT LAYER

 x₁ ─────────────► ○
 x₂ ─────────────► ○ ───────────────► ○  class 1
 x₃ ─────────────► ○ ───────────────► ○  class 2
 x₄ ─────────────► ○ ───────────────► ○  class 3
                  ○
                  ○
                  ○

4 features        6 neurons          3 outputs
```

Una forma compacta de describirla es:

```text
4 → 6 → 3
```


## 7. ¿Qué significa "Deep"?

Una red con una hidden layer:

```text
4 → 6 → 3
```

es relativamente sencilla.

Una red más profunda podría ser:

```text
4 → 16 → 16 → 8 → 3
```

Cada flecha representa una transformación:

\[
\mathbf{z}^{(\ell)} =
W^{(\ell)}\mathbf{a}^{(\ell-1)} + b^{(\ell)}.
\]

seguida normalmente por una activación.

```text
Input
  ↓
Linear
  ↓
Activation
  ↓
Linear
  ↓
Activation
  ↓
...
  ↓
Output
```


## 8. Calculando parámetros

Para una capa totalmente conectada con:

```text
n_inputs → n_outputs
```

el número de parámetros es:

\[
(n_{\mathrm{inputs}}\times n_{\mathrm{outputs}})
+
n_{\mathrm{outputs}}.
\]

El primer término corresponde a weights.

El segundo corresponde a biases.


### Reto

¿Cuántos parámetros tiene:

```text
4 → 8
```

?

<details>
<summary><strong>Pista</strong></summary>

\[
4\times8+8
\]

</details>

<details>
<summary><strong>Mostrar solución</strong></summary>

\[
32+8=40
\]

parámetros.

</details>


### Reto 2

¿Cuántos parámetros tiene toda esta red?

```text
4 → 8 → 3
```

<details>
<summary><strong>Pista</strong></summary>

Calcula cada capa por separado.

</details>

<details>
<summary><strong>Mostrar solución</strong></summary>

Primera capa:

\[
4\times8+8=40
\]

Segunda capa:

\[
8\times3+3=27
\]

Total:

\[
40+27=67
\]

parámetros.

</details>


## 9. Implementemos una red pequeña desde cero

Construiremos:

```text
4 → 5 → 3
```

sin PyTorch todavía.


In [ ]:
rng = np.random.default_rng(42)

W1 = rng.normal(size=(5, 4))
b1 = rng.normal(size=5)

W2 = rng.normal(size=(3, 5))
b2 = rng.normal(size=3)

def forward(x):
    z1 = W1 @ x + b1
    a1 = relu(z1)

    z2 = W2 @ a1 + b2

    return z2

x = np.array([5.1, 3.5, 1.4, 0.2])

output = forward(x)

print(output)


## 10. Ejemplo científico: arquitectura para estrellas

Supongamos que tenemos:

```text
temperature
luminosity
radius
color_index
metallicity
```

Eso da:

\[
5\text{ features}.
\]

Queremos clasificar una estrella en 4 categorías.

Una arquitectura posible:

```text
5 → 12 → 8 → 4
```

Interpretación:

```text
5 mediciones científicas
        ↓
12 neuronas
        ↓
8 neuronas
        ↓
4 categorías
```

La arquitectura es una decisión de diseño.

No existe una única arquitectura correcta para todos los problemas.


# Ejercicio interactivo

Queremos construir una red para un problema meteorológico con:

- temperatura;
- humedad;
- presión;
- velocidad del viento;
- precipitación previa;
- radiación solar.

Queremos predecir 3 clases:

```text
clear
rain
storm
```

### Pregunta

Propón una arquitectura pequeña.

Escribe algo como:

```text
6 → ? → ? → 3
```

<details>
<summary><strong>Pista</strong></summary>

Puedes comenzar con algo sencillo como 8 o 16 neuronas por hidden layer.

</details>

<details>
<summary><strong>Mostrar una posible solución</strong></summary>

Por ejemplo:

```text
6 → 12 → 8 → 3
```

No es la única respuesta correcta.

Después necesitaremos entrenar y evaluar la red para saber si esa arquitectura funciona bien.

</details>


# Para recordar

Una arquitectura describe cómo fluye la información:

```text
features
   ↓
hidden layers
   ↓
output
```

Por ejemplo:

```text
4 → 8 → 8 → 3
```

nos dice:

- 4 inputs;
- 2 hidden layers de 8 neuronas;
- 3 outputs.

Ya sabemos:

1. qué hace una neurona;
2. qué son weights y bias;
3. qué hacen las activaciones;
4. cómo conectar neuronas en capas.

El siguiente gran paso será aprender:

# ¿Cómo se entrena una red neuronal?
